# Data Cleaning

In [58]:
import pandas as pd

In [59]:
df = pd.read_csv("../data/reviews_badminton/data.csv")

In [60]:
df.head()

,Reviewer Name,Review Title,Place of Review,Up Votes,Down Votes,Month,Review text,Ratings
0,Kamal Suresh,Nice product,"Certified Buyer, Chirakkal",889.0,64.0,Feb 2021,"Nice product, good quality, but price is now r...",4
1,Flipkart Customer,Don't waste your money,"Certified Buyer, Hyderabad",109.0,6.0,Feb 2021,They didn't supplied Yonex Mavis 350. Outside ...,1
2,A. S. Raja Srinivasan,Did not meet expectations,"Certified Buyer, Dharmapuri",42.0,3.0,Apr 2021,Worst product. Damaged shuttlecocks packed in ...,1
3,Suresh Narayanasamy,Fair,"Certified Buyer, Chennai",25.0,1.0,NaN,"Quite O. K. , but nowadays the quality of the...",3
4,ASHIK P A,Over priced,NaN,147.0,24.0,Apr 2016,Over pricedJust â?¹620 ..from retailer.I didn'...,1


In [61]:
df.columns

Index(['Reviewer Name', 'Review Title', 'Place of Review', 'Up Votes',
       'Down Votes', 'Month', 'Review text', 'Ratings'],
      dtype='object')

In [62]:
df = df[['Review text', 'Ratings']]

df.head()

,Review text,Ratings
0,"Nice product, good quality, but price is now r...",4
1,They didn't supplied Yonex Mavis 350. Outside ...,1
2,Worst product. Damaged shuttlecocks packed in ...,1
3,"Quite O. K. , but nowadays the quality of the...",3
4,Over pricedJust â?¹620 ..from retailer.I didn'...,1


In [63]:
def label_sentiment(rating):
    if rating >= 3:
        return 1      # Positive
    else:
        return 0      # Negative

df['Sentiment'] = df['Ratings'].apply(label_sentiment)

df.head()

,Review text,Ratings,Sentiment
0,"Nice product, good quality, but price is now r...",4,1
1,They didn't supplied Yonex Mavis 350. Outside ...,1,0
2,Worst product. Damaged shuttlecocks packed in ...,1,0
3,"Quite O. K. , but nowadays the quality of the...",3,1
4,Over pricedJust â?¹620 ..from retailer.I didn'...,1,0


In [64]:
df['Sentiment'].value_counts()

Sentiment
1    7441
0    1077
Name: count, dtype: int64

In [65]:
df.isnull().sum()

Review text    8
Ratings        0
Sentiment      0
dtype: int64

In [66]:
df.dropna(inplace=True)

In [67]:
df.isnull().sum()

Review text    0
Ratings        0
Sentiment      0
dtype: int64

In [68]:
!pip install nltk


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [69]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\dharm\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\dharm\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\dharm\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [70]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()                           # lowercase
    text = re.sub(r'[^a-z\s]', '', text)          # remove special characters and numbers
    words = text.split()
    
    words = [lemmatizer.lemmatize(word) 
             for word in words if word not in stop_words]
    
    return " ".join(words)

In [71]:
df['clean_text'] = df['Review text'].apply(clean_text)

df.head()

,Review text,Ratings,Sentiment,clean_text
0,"Nice product, good quality, but price is now r...",4,1,nice product good quality price rising bad sig...
1,They didn't supplied Yonex Mavis 350. Outside ...,1,0,didnt supplied yonex mavis outside cover yonex...
2,Worst product. Damaged shuttlecocks packed in ...,1,0,worst product damaged shuttlecock packed new b...
3,"Quite O. K. , but nowadays the quality of the...",3,1,quite k nowadays quality cork like year back u...
4,Over pricedJust â?¹620 ..from retailer.I didn'...,1,0,pricedjust retaileri didnt understand wat adva...


In [72]:
final_df = df[['clean_text', 'Sentiment']]

final_df.head()


,clean_text,Sentiment
0,nice product good quality price rising bad sig...,1
1,didnt supplied yonex mavis outside cover yonex...,0
2,worst product damaged shuttlecock packed new b...,0
3,quite k nowadays quality cork like year back u...,1
4,pricedjust retaileri didnt understand wat adva...,0


In [73]:
final_df.to_csv("../data/clean_reviews.csv", index=False)

# Model Creation

In [74]:
import pandas as pd

df = pd.read_csv("../data/clean_reviews.csv")
df.head()

,clean_text,Sentiment
0,nice product good quality price rising bad sig...,1
1,didnt supplied yonex mavis outside cover yonex...,0
2,worst product damaged shuttlecock packed new b...,0
3,quite k nowadays quality cork like year back u...,1
4,pricedjust retaileri didnt understand wat adva...,0


In [75]:
X = df['clean_text']
y = df['Sentiment']

In [76]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [77]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

## (A) Logistic Regression

In [78]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

lr = LogisticRegression()
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

print("F1 Score (Logistic Regression):", f1_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

F1 Score (Logistic Regression): 0.9555915721231767
              precision    recall  f1-score   support

           0       0.87      0.43      0.57       214
           1       0.92      0.99      0.96      1488

    accuracy                           0.92      1702
   macro avg       0.89      0.71      0.76      1702
weighted avg       0.92      0.92      0.91      1702



In [79]:
import pickle

pickle.dump(tfidf, open("../models/tfidf_vectorizer.pkl", "wb"))
pickle.dump(lr, open("../models/sentiment_model.pkl", "wb"))

## (B) Naive Bayes

In [80]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

y_pred_nb = nb.predict(X_test_tfidf)

print("F1 Score (Naive Bayes):", f1_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

F1 Score (Naive Bayes): 0.9440559440559441
              precision    recall  f1-score   support

           0       0.93      0.19      0.32       214
           1       0.90      1.00      0.94      1488

    accuracy                           0.90      1702
   macro avg       0.91      0.59      0.63      1702
weighted avg       0.90      0.90      0.87      1702



## (C) Support Vector Machine

In [81]:
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(X_train_tfidf, y_train)

y_pred_svm = svm.predict(X_test_tfidf)

print("F1 Score (SVM):", f1_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))


F1 Score (SVM): 0.9537953795379538
              precision    recall  f1-score   support

           0       0.73      0.55      0.63       214
           1       0.94      0.97      0.95      1488

    accuracy                           0.92      1702
   macro avg       0.83      0.76      0.79      1702
weighted avg       0.91      0.92      0.91      1702

